# MRI masked autoencoder: a minimal baseline

We first learn to reconstruct hidden MRI patches **without diagnoses**, then freeze
the encoder and fit regularized logistic classifiers on the labeled studies.
All planes and series are mixed. No reports, pretrained weights, or test images
are used. This notebook contains the model and training code; no project module
or package download is required.

**Start with Run All in the default `smoke` mode.** It runs a few batches with the
same model, then exercises all four classifier comparisons. Its scores are
**pipeline checks, not performance estimates**. For the real pilot, set
`MODE = "pilot"` and `ALLOW_PILOT = True`, run through the benchmark, inspect the
estimated duration, then run the training cells. Nothing trains just by opening
the notebook. Dependencies are listed in the project's `requirements.txt`.

On Colab, copy/extract the data to runtime-local storage (for example `/content/data`)
and change `DATA_DIR`. Reading thousands of small JPEGs from mounted Drive can be
slow. Outputs on a Colab runtime are ephemeral: copy checkpoints to persistent
storage between sessions. No installation, download, or copy is automatic.

The full workflow is: **inspect → split → index/validate → MAE → study features →
five-fold linear probes → compare**. Study grouping is available; these CSVs do
not establish patient-level independence.

In [ ]:
from pathlib import Path
import copy
import hashlib
import json
import math
import platform
import random
import time
import warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import display
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from tqdm.auto import tqdm

# Resolve the repository from the root or this notebook's folder.
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "baseline_candidate").is_dir() and (p / "notebooks").is_dir()), Path.cwd())

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_ROOT = PROJECT_ROOT / "outputs/mae_baseline"
MODE = "smoke"                       # "smoke" or "pilot"
ALLOW_PILOT = False                   # explicit opt-in to the full 30-epoch pilot
RUN_NAME = None                       # None generates a unique timestamped name
RESUME_RUN = None                     # existing run directory; resume SAME config
NUM_WORKERS = 0 if platform.system() != "Linux" else 2
NORM_PIX_LOSS = True
SEED = 42

assert MODE in {"smoke", "pilot"}
if MODE == "pilot" and not ALLOW_PILOT:
    raise RuntimeError("Set ALLOW_PILOT=True after choosing the pilot configuration.")
if NUM_WORKERS and platform.system() != "Linux":
    raise ValueError("Use NUM_WORKERS=0 in this self-contained macOS notebook (spawn pickling).")

CFG = dict(
    mode=MODE, seed=SEED, image_size=224, patch_size=16, mask_ratio=0.75,
    encoder_dim=192, encoder_depth=4, encoder_heads=3,
    decoder_dim=128, decoder_depth=2, decoder_heads=4, mlp_ratio=4,
    norm_pix_loss=NORM_PIX_LOSS, epochs=2 if MODE == "smoke" else 30,
    batch_size=8 if MODE == "smoke" else 32,
    samples_per_study=2 if MODE == "smoke" else 4,
    validation_samples=2 if MODE == "smoke" else 4,
    peak_lr=1e-4, min_lr=1e-6, warmup_epochs=1, weight_decay=0.05,
    betas=[0.9, 0.95], num_workers=NUM_WORKERS, n_folds=5, probe_C=0.1,
    probe_max_iter=100, probe_grad_tol=1e-6, probe_change_tol=1e-15,
    smoke_train_studies=8, smoke_val_studies=4, smoke_images_per_study=2,
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
# Avoid CPU thread overhead in the tiny linear probes and smoke run.
torch.set_num_threads(min(4, torch.get_num_threads()))

if RESUME_RUN is not None:
    RUN_DIR = Path(RESUME_RUN)
    assert RUN_DIR.is_dir(), RUN_DIR
    saved_cfg = json.loads((RUN_DIR / "config.json").read_text())
    assert saved_cfg == CFG, "Resume requires the original configuration, including duration."
else:
    run_name = RUN_NAME or f"{MODE}_{datetime.now(timezone.utc):%Y%m%dT%H%M%S_%fZ}"
    RUN_DIR = OUTPUT_ROOT / run_name
    RUN_DIR.mkdir(parents=True, exist_ok=False)
    (RUN_DIR / "config.json").write_text(json.dumps(CFG, indent=2))

(RUN_DIR / "environment.json").write_text(json.dumps({
    "torch": torch.__version__, "numpy": np.__version__, "pandas": pd.__version__,
    "platform": platform.platform(), "device": str(DEVICE),
    "data_dir": str(DATA_DIR.resolve()),
}, indent=2))
print(f"Mode: {MODE} | device: {DEVICE} | outputs: {RUN_DIR}")
if MODE == "smoke":
    print("SMOKE TEST ONLY: reduced images and training; scores are not performance estimates.")
display(pd.Series(CFG, name="value").to_frame())

## 1. Split by study before touching image features

All labeled studies stay out of MAE training and reconstruction validation.
Missing labels are not zeros. Partial or non-binary label rows stop execution.
The unlabeled split is fixed at seed 42. Smoke mode takes a small subset of this
same split and only two images per active study, including the labeled studies.
The pilot indexes and validates every training JPEG before optimization.

In [ ]:
LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
          "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
train = pd.read_csv(DATA_DIR / "train.csv", dtype={"StudyInstanceUID": str})
assert not train.StudyInstanceUID.duplicated().any(), "Duplicate study rows"
y_all = train[LABELS].apply(pd.to_numeric, errors="raise")
known = y_all.notna().sum(axis=1)
assert known.isin([0, len(LABELS)]).all(), "Partially labeled studies need an explicit policy"
assert (y_all.isin([0, 1]) | y_all.isna()).to_numpy().all(), "Labels must be 0, 1, or missing"
labeled = train.loc[known == len(LABELS), ["StudyInstanceUID"] + LABELS].copy()
labeled = labeled.sort_values("StudyInstanceUID").reset_index(drop=True)
unlabeled_ids = sorted(train.loc[known == 0, "StudyInstanceUID"].tolist())
mae_train_ids, mae_val_ids = train_test_split(unlabeled_ids, test_size=0.05, random_state=SEED)
mae_train_ids, mae_val_ids = sorted(mae_train_ids), sorted(mae_val_ids)
labeled_ids = labeled.StudyInstanceUID.tolist()
assert len(labeled) >= CFG["n_folds"]
assert not (set(mae_train_ids) & set(mae_val_ids))
assert not ((set(mae_train_ids) | set(mae_val_ids)) & set(labeled_ids))
test_ids = set(pd.read_csv(DATA_DIR / "test.csv", dtype=str).StudyInstanceUID)
assert not (set(train.StudyInstanceUID) & test_ids), "Train/test study overlap"

available_ids = {p.name for p in (DATA_DIR / "train_series").iterdir() if p.is_dir()}
missing = set(train.StudyInstanceUID) - available_ids
assert not missing, f"Missing image directories: {sorted(missing)[:5]}"
extra = available_ids - set(train.StudyInstanceUID)
if extra:
    print(f"Ignoring {len(extra)} study directories absent from train.csv")

active_train_ids = mae_train_ids
active_val_ids = mae_val_ids
if MODE == "smoke":
    rng = np.random.default_rng(SEED)
    active_train_ids = sorted(rng.choice(mae_train_ids, CFG["smoke_train_studies"], replace=False))
    active_val_ids = sorted(rng.choice(mae_val_ids, CFG["smoke_val_studies"], replace=False))
active_ids = set(active_train_ids) | set(active_val_ids) | set(labeled_ids)
split_of = {s: "mae_train" for s in mae_train_ids}
split_of.update({s: "mae_validation" for s in mae_val_ids})
split_of.update({s: "labeled" for s in labeled_ids})
splits = pd.DataFrame([
    {"study_id": s, "split": split_of[s], "active": s in active_ids}
    for s in sorted(split_of)
])
split_csv = splits.to_csv(index=False)
if RESUME_RUN is not None:
    assert (RUN_DIR / "splits.csv").read_text() == split_csv, "Dataset split changed"
else:
    (RUN_DIR / "splits.csv").write_text(split_csv)
print(f"Labeled: {len(labeled)} | unlabeled: {len(unlabeled_ids)}")
display(splits.groupby(["split", "active"]).size().rename("studies").to_frame())
display(pd.DataFrame({"positive": labeled[LABELS].sum().astype(int),
                      "negative": (1 - labeled[LABELS]).sum().astype(int)}))

In [ ]:
def stable_seed(text, offset=0):
    return (SEED + offset + int(hashlib.sha256(str(text).encode()).hexdigest()[:8], 16)) % 2**32


records = []
for study_id in tqdm(sorted(active_ids), desc="Indexing studies"):
    paths = sorted((DATA_DIR / "train_series" / study_id).glob("*/*.jpeg"))
    if not paths:
        raise ValueError(f"No JPEGs for study {study_id}")
    if MODE == "smoke" and len(paths) > CFG["smoke_images_per_study"]:
        rng = np.random.default_rng(stable_seed(study_id))
        paths = [paths[i] for i in sorted(rng.choice(len(paths), CFG["smoke_images_per_study"], replace=False))]
    for path in paths:
        stat = path.stat()
        records.append(dict(study_id=study_id, series_id=path.parent.name,
                            path=path.relative_to(DATA_DIR).as_posix(), split=split_of[study_id],
                            size_bytes=stat.st_size, mtime_ns=stat.st_mtime_ns))
manifest = pd.DataFrame(records)
assert not manifest.path.duplicated().any()
manifest_csv = manifest.to_csv(index=False)
manifest_hash = hashlib.sha256(manifest_csv.encode()).hexdigest()
if RESUME_RUN is not None:
    assert (RUN_DIR / "manifest.sha256").read_text() == manifest_hash, "Image manifest changed"
else:
    (RUN_DIR / "manifest.csv").write_text(manifest_csv)
    (RUN_DIR / "manifest.sha256").write_text(manifest_hash)
study_paths = {s: rows.path.tolist() for s, rows in manifest.groupby("study_id", sort=True)}
del records, manifest_csv


def read_image(relative_path):
    path = DATA_DIR / relative_path
    with Image.open(path) as im:
        if im.size != (CFG["image_size"], CFG["image_size"]) or im.mode != "L":
            raise ValueError(f"Expected grayscale 224x224 JPEG, got {im.mode} {im.size}: {path}")
        array = np.array(im, dtype=np.float32, copy=True) / 255.0
    return torch.from_numpy(array).unsqueeze(0)


errors = []
for path in tqdm(manifest.path, desc="Validating every indexed image"):
    try:
        read_image(path)
    except Exception as exc:
        errors.append({"path": path, "error": str(exc)})
if errors:
    pd.DataFrame(errors).to_csv(RUN_DIR / "image_errors.csv", index=False)
    raise RuntimeError(f"{len(errors)} invalid images; see image_errors.csv. No silent skipping.")
print(f"Validated {len(manifest):,} JPEGs across {len(study_paths)} active studies.")
fig, axes = plt.subplots(1, 4, figsize=(10, 3))
for ax, path in zip(axes, manifest.path.iloc[:4]):
    ax.imshow(read_image(path)[0], cmap="gray", vmin=0, vmax=1)
    ax.axis("off")
fig.suptitle(f"Input images — {MODE}")
fig.tight_layout()
fig.savefig(RUN_DIR / "input_examples.png", dpi=130)
plt.show()

## 2. Small masked autoencoder

A 224 × 224 image contains 196 patches. At 75% masking, the encoder sees 49;
the decoder receives those embeddings plus 147 mask tokens restored to their
original spatial positions. Fixed positional embeddings supply patch locations.
No class token is needed: downstream image features average all patch embeddings.

The loss compares **only masked patches** with targets normalized independently
within each patch: `(patch - mean) / sqrt(variance + 1e-6)`. Population variance
is used consistently. This removes patch-level brightness and scale from the
target; it does not guarantee that learned features encode pathology.

In [ ]:
def patchify(x, patch_size=16):
    b, c, h, w = x.shape
    assert c == 1 and h == w and h % patch_size == 0
    g = h // patch_size
    return x.reshape(b, c, g, patch_size, g, patch_size).permute(0, 2, 4, 3, 5, 1).reshape(b, g*g, patch_size**2)


def unpatchify(patches, patch_size=16):
    b, n, d = patches.shape
    g = math.isqrt(n)
    assert g*g == n and d == patch_size**2
    return patches.reshape(b, g, g, patch_size, patch_size, 1).permute(0, 5, 1, 3, 2, 4).reshape(b, 1, g*patch_size, g*patch_size)


def normalized_targets(patches):
    mean = patches.mean(dim=-1, keepdim=True)
    scale = (patches.var(dim=-1, keepdim=True, unbiased=False) + 1e-6).sqrt()
    return (patches - mean) / scale, mean, scale


def sincos_2d(grid, dim):
    assert dim % 4 == 0
    y, x = torch.meshgrid(torch.arange(grid), torch.arange(grid), indexing="ij")
    freq = 1.0 / (10000 ** (torch.arange(dim // 4, dtype=torch.float32) / (dim // 4)))
    embeddings = []
    for coord in (y.flatten(), x.flatten()):
        phase = coord.float()[:, None] * freq[None, :]
        embeddings.extend([phase.sin(), phase.cos()])
    return torch.cat(embeddings, dim=-1).unsqueeze(0)


class Block(nn.Module):
    def __init__(self, dim, heads, ratio):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, dropout=0.0, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, ratio*dim), nn.GELU(), nn.Linear(ratio*dim, dim))

    def forward(self, x):
        z = self.norm1(x)
        x = x + self.attn(z, z, z, need_weights=False)[0]
        return x + self.mlp(self.norm2(x))


class SmallMAE(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.patch_size = cfg["patch_size"]
        self.mask_ratio = cfg["mask_ratio"]
        self.norm_pix_loss = cfg["norm_pix_loss"]
        e, d = cfg["encoder_dim"], cfg["decoder_dim"]
        grid = cfg["image_size"] // self.patch_size
        self.patch_embed = nn.Linear(self.patch_size**2, e)
        self.register_buffer("encoder_pos", sincos_2d(grid, e))
        self.encoder = nn.Sequential(*[Block(e, cfg["encoder_heads"], cfg["mlp_ratio"]) for _ in range(cfg["encoder_depth"])])
        self.encoder_norm = nn.LayerNorm(e)
        self.decoder_embed = nn.Linear(e, d)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, d))
        self.register_buffer("decoder_pos", sincos_2d(grid, d))
        self.decoder = nn.Sequential(*[Block(d, cfg["decoder_heads"], cfg["mlp_ratio"]) for _ in range(cfg["decoder_depth"])])
        self.decoder_norm = nn.LayerNorm(d)
        self.decoder_pred = nn.Linear(d, self.patch_size**2)
        self.apply(self._initialize)
        nn.init.normal_(self.mask_token, std=0.02)

    @staticmethod
    def _initialize(module):
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.LayerNorm):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)

    def random_mask(self, tokens, noise=None):
        b, n, dim = tokens.shape
        # CPU-generated randomness works identically with CPU, CUDA and MPS.
        if noise is None:
            noise = torch.rand(b, n)
        ids_shuffle = noise.to(tokens.device).argsort(dim=1)
        ids_restore = ids_shuffle.argsort(dim=1)
        keep = int(n * (1 - self.mask_ratio))
        ids_keep = ids_shuffle[:, :keep]
        visible = tokens.gather(1, ids_keep.unsqueeze(-1).expand(-1, -1, dim))
        mask = torch.ones(b, n, device=tokens.device)
        mask[:, :keep] = 0
        mask = mask.gather(1, ids_restore)
        return visible, mask, ids_restore, ids_keep

    def encode_images(self, x):
        tokens = self.patch_embed(patchify(x, self.patch_size)) + self.encoder_pos
        return self.encoder_norm(self.encoder(tokens)).mean(dim=1)

    def forward(self, x, noise=None):
        patches = patchify(x, self.patch_size)
        tokens = self.patch_embed(patches) + self.encoder_pos
        visible, mask, ids_restore, _ = self.random_mask(tokens, noise)
        latent = self.encoder_norm(self.encoder(visible))
        decoded = self.decoder_embed(latent)
        missing = patches.shape[1] - decoded.shape[1]
        decoded = torch.cat([decoded, self.mask_token.expand(x.shape[0], missing, -1)], dim=1)
        decoded = decoded.gather(1, ids_restore.unsqueeze(-1).expand(-1, -1, decoded.shape[-1]))
        prediction = self.decoder_pred(self.decoder_norm(self.decoder(decoded + self.decoder_pos)))
        target = normalized_targets(patches)[0] if self.norm_pix_loss else patches
        patch_loss = (prediction - target).square().mean(dim=-1)
        loss = (patch_loss * mask).sum() / mask.sum()
        return loss, prediction, mask


torch.manual_seed(SEED)
model = SmallMAE(CFG).to(DEVICE)
initial_path = RUN_DIR / "initialization.pt"
if RESUME_RUN is not None:
    model.load_state_dict(torch.load(initial_path, map_location=DEVICE, weights_only=True))
else:
    torch.save({k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, initial_path)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## 3. Sampling, schedule, and a timing estimate

Training samples each study equally, then samples images uniformly within that
study. Images are resampled every epoch; series with more images can contribute
more samples. Validation images and masks stay fixed. If a study has fewer
images than requested, sampling uses replacement explicitly.

The benchmark includes loading plus a forward/backward pass and synchronizes the
accelerator. It restores initial weights and the random-number state afterward.
It estimates training time, not the complete feature-extraction/CV runtime.

In [ ]:
class ImageDataset(Dataset):
    def __init__(self, paths, fixed_masks=False):
        self.paths = list(paths)
        self.fixed_masks = fixed_masks

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        path = self.paths[index]
        x = read_image(path)
        if not self.fixed_masks:
            return x
        generator = torch.Generator().manual_seed(stable_seed(path, 999))
        n = (CFG["image_size"] // CFG["patch_size"])**2
        return x, torch.rand(n, generator=generator)


def sampled_paths(ids, count, epoch):
    rng = np.random.default_rng(SEED + epoch)
    selected = []
    for study_id in sorted(ids):
        paths = study_paths[study_id]
        indices = rng.choice(len(paths), count, replace=len(paths) < count)
        selected.extend(paths[i] for i in indices)
    rng.shuffle(selected)
    return selected


def make_loader(paths, fixed_masks=False):
    kwargs = {}
    if NUM_WORKERS:
        # Dataset/functions live in this notebook; Linux fork avoids spawn pickling.
        # Workers only decode CPU images and never touch the accelerator.
        kwargs["multiprocessing_context"] = "fork"
    return DataLoader(ImageDataset(paths, fixed_masks), batch_size=CFG["batch_size"],
                      shuffle=False, drop_last=False, num_workers=NUM_WORKERS,
                      pin_memory=DEVICE.type == "cuda",
                      generator=torch.Generator().manual_seed(SEED), **kwargs)


validation_paths = sampled_paths(active_val_ids, CFG["validation_samples"], epoch=10000)
pd.DataFrame({"path": validation_paths}).to_csv(RUN_DIR / "validation_images.csv", index=False)
steps_per_epoch = math.ceil(len(active_train_ids) * CFG["samples_per_study"] / CFG["batch_size"])
total_steps = CFG["epochs"] * steps_per_epoch
warmup_steps = CFG["warmup_epochs"] * steps_per_epoch


def lr_at_step(step):
    if step < warmup_steps:
        return CFG["peak_lr"] * (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps - 1)
    progress = min(max(progress, 0.0), 1.0)
    return CFG["min_lr"] + 0.5 * (CFG["peak_lr"] - CFG["min_lr"]) * (1 + math.cos(math.pi * progress))


def make_optimizer(net):
    decay, no_decay = [], []
    for name, parameter in net.named_parameters():
        if not parameter.requires_grad:
            continue
        (no_decay if parameter.ndim <= 1 or name.endswith("bias") else decay).append(parameter)
    return torch.optim.AdamW([
        {"params": decay, "weight_decay": CFG["weight_decay"]},
        {"params": no_decay, "weight_decay": 0.0},
    ], lr=CFG["peak_lr"], betas=tuple(CFG["betas"]))


def synchronize():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    elif DEVICE.type == "mps":
        torch.mps.synchronize()


def capture_rng():
    result = {"python": random.getstate(), "numpy": np.random.get_state(), "torch": torch.get_rng_state()}
    if DEVICE.type == "cuda":
        result["cuda"] = torch.cuda.get_rng_state_all()
    if DEVICE.type == "mps":
        result["mps"] = torch.mps.get_rng_state()
    return result


def restore_rng(state):
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"].cpu())
    if DEVICE.type == "cuda" and "cuda" in state:
        torch.cuda.set_rng_state_all([s.cpu() for s in state["cuda"]])
    if DEVICE.type == "mps" and "mps" in state:
        torch.mps.set_rng_state(state["mps"].cpu())


bench_rng = capture_rng()
benchmark_paths = sampled_paths(active_train_ids, CFG["samples_per_study"], 0)[:CFG["batch_size"] * 4]
start = time.perf_counter()
benchmark_batches = list(make_loader(benchmark_paths))
load_seconds = time.perf_counter() - start
benchmark_optimizer = make_optimizer(model)
model.train()
x = benchmark_batches[0].to(DEVICE)
loss, _, _ = model(x)
loss.backward()
benchmark_optimizer.zero_grad(set_to_none=True)
synchronize()
start = time.perf_counter()
for x in benchmark_batches:
    x = x.to(DEVICE)
    benchmark_optimizer.zero_grad(set_to_none=True)
    loss, _, _ = model(x)
    loss.backward()
    benchmark_optimizer.step()
synchronize()
compute_seconds = time.perf_counter() - start
seconds_per_batch = (load_seconds + compute_seconds) / len(benchmark_batches)
benchmark = dict(load_seconds=load_seconds, compute_seconds=compute_seconds,
                 batches=len(benchmark_batches), images=len(benchmark_paths),
                 estimated_training_epoch_seconds=seconds_per_batch * steps_per_epoch)
print(json.dumps(benchmark, indent=2))
print("Approximate estimate; storage caching and worker overlap affect actual timing.")
(RUN_DIR / "benchmark.json").write_text(json.dumps(benchmark, indent=2))
model.load_state_dict(torch.load(initial_path, map_location=DEVICE, weights_only=True))
model.zero_grad(set_to_none=True)
restore_rng(bench_rng)
del benchmark_optimizer, benchmark_batches, x, loss

fig, ax = plt.subplots(figsize=(6, 2))
ax.plot([lr_at_step(i) for i in range(total_steps)])
ax.set(xlabel="Optimizer step", ylabel="Learning rate", title="Planned warmup + cosine schedule")
fig.tight_layout()
fig.savefig(RUN_DIR / "learning_rate.png", dpi=130)
plt.show()

## 4. Verify the model before training

These assertions check the actual masking, patch operations, loss, finite
gradients, and schedule. They restore the initial model and RNG afterward.

In [ ]:
check_rng = capture_rng()
x_check = torch.stack([read_image(validation_paths[0]), read_image(validation_paths[-1])]).to(DEVICE)
p = patchify(x_check)
assert torch.equal(unpatchify(p), x_check), "Patch round trip failed"
constant_norm, _, _ = normalized_targets(torch.ones(2, 196, 256, device=DEVICE))
assert torch.isfinite(constant_norm).all() and torch.count_nonzero(constant_norm) == 0
tokens = torch.arange(196, device=DEVICE).view(1, 196, 1).float().expand(2, -1, 3)
noise = torch.rand(2, 196, generator=torch.Generator().manual_seed(123))
visible, mask, restore, keep = model.random_mask(tokens, noise)
assert visible.shape[1] == 49 and (mask.sum(dim=1) == 147).all()
assert (mask.gather(1, keep) == 0).all()
restored = torch.cat([visible, torch.full((2, 147, 3), -1., device=DEVICE)], dim=1)
restored = restored.gather(1, restore.unsqueeze(-1).expand(-1, -1, 3))
assert torch.equal(restored[mask == 0], tokens[mask == 0])
assert (restored[mask == 1] == -1).all()
model.train()
loss, prediction, mask = model(x_check, noise)
target = normalized_targets(p)[0] if NORM_PIX_LOSS else p
manual_loss = ((prediction - target).square().mean(-1) * mask).sum() / mask.sum()
assert torch.allclose(loss, manual_loss)
perturbed = prediction.detach().clone()
perturbed[mask == 0] += 100
assert torch.allclose(((perturbed - target).square().mean(-1) * mask).sum() / mask.sum(), loss.detach())
loss.backward()
assert torch.isfinite(loss)
assert all(torch.isfinite(p.grad).all() for p in model.parameters() if p.grad is not None)
assert any(p.grad is not None and p.grad.abs().sum() > 0 for p in model.parameters())
assert math.isclose(lr_at_step(warmup_steps - 1), CFG["peak_lr"])
assert math.isclose(lr_at_step(total_steps - 1), CFG["min_lr"])
schedule = np.array([lr_at_step(i) for i in range(total_steps)])
assert (np.diff(schedule[:warmup_steps]) >= 0).all()
assert (np.diff(schedule[warmup_steps:]) <= 1e-15).all()
model.load_state_dict(torch.load(initial_path, map_location=DEVICE, weights_only=True))
model.zero_grad(set_to_none=True)
restore_rng(check_rng)
del tokens, visible, restored, prediction, perturbed, target, loss, p
print("Model, masking, loss, gradients and schedule checks passed.")

## 5. Train the MAE

This cell launches the selected run. The default smoke run takes only a few
optimizer steps. Validation selects the checkpoint using reconstruction loss,
never the 58 diagnostic labels. Checkpoints resume at completed epoch boundaries.
To resume, set `RESUME_RUN` to the existing output directory and retain its exact
configuration, then rerun the notebook. Only load checkpoints you trust.

In [ ]:
def atomic_torch_save(obj, path):
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, temporary)
    temporary.replace(path)


optimizer = make_optimizer(model)
history, start_epoch, global_step, best_validation = [], 0, 0, float("inf")
latest_path, best_path = RUN_DIR / "latest.pt", RUN_DIR / "best.pt"
if RESUME_RUN is not None:
    # weights_only=False is needed for Python/NumPy RNG states; trusted local file only.
    saved = torch.load(latest_path, map_location="cpu", weights_only=False)
    assert saved["config"] == CFG and saved["manifest_hash"] == manifest_hash
    model.load_state_dict(saved["model"])
    optimizer.load_state_dict(saved["optimizer"])
    start_epoch, global_step = saved["next_epoch"], saved["global_step"]
    best_validation, history = saved["best_validation"], saved["history"]
    assert global_step == start_epoch * steps_per_epoch
    restore_rng(saved["rng"])
    print(f"Resuming at epoch {start_epoch + 1}, optimizer step {global_step}")

for epoch in range(start_epoch, CFG["epochs"]):
    epoch_paths = sampled_paths(active_train_ids, CFG["samples_per_study"], epoch)
    model.train()
    total_loss, image_count = 0.0, 0
    synchronize()
    started = time.perf_counter()
    for images in tqdm(make_loader(epoch_paths), desc=f"MAE epoch {epoch + 1}/{CFG['epochs']}"):
        images = images.to(DEVICE)
        lr = lr_at_step(global_step)
        for group in optimizer.param_groups:
            group["lr"] = lr
        optimizer.zero_grad(set_to_none=True)
        loss, _, _ = model(images)
        if not torch.isfinite(loss):
            raise RuntimeError(f"Nonfinite training loss at step {global_step}")
        loss.backward()
        if not torch.stack([torch.isfinite(p.grad).all() for p in model.parameters() if p.grad is not None]).all():
            raise RuntimeError(f"Nonfinite gradients at step {global_step}")
        optimizer.step()
        total_loss += loss.item() * len(images)
        image_count += len(images)
        global_step += 1
    model.eval()
    val_loss, val_count = 0.0, 0
    with torch.no_grad():
        for images, noise in make_loader(validation_paths, fixed_masks=True):
            loss, _, _ = model(images.to(DEVICE), noise)
            if not torch.isfinite(loss):
                raise RuntimeError("Nonfinite validation loss")
            val_loss += loss.item() * len(images)
            val_count += len(images)
    val_loss /= val_count
    synchronize()
    history.append(dict(epoch=epoch + 1, train_loss=total_loss / image_count,
                        validation_loss=val_loss, last_lr=lr,
                        seconds=time.perf_counter() - started, global_step=global_step))
    improved = val_loss < best_validation
    best_validation = min(best_validation, val_loss)
    state = dict(model={k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                 optimizer=optimizer.state_dict(), config=CFG, manifest_hash=manifest_hash,
                 next_epoch=epoch + 1, global_step=global_step, best_validation=best_validation,
                 history=history, rng=capture_rng())
    atomic_torch_save(state, latest_path)
    if improved:
        atomic_torch_save(state, best_path)
    pd.DataFrame(history).to_csv(RUN_DIR / "history.csv", index=False)
    print(history[-1])

best = torch.load(best_path, map_location="cpu", weights_only=False)
model.load_state_dict(best["model"])
model.eval()
with torch.no_grad():
    reference_features = model.encode_images(x_check)
reload_model = SmallMAE(CFG).to(DEVICE)
reload_model.load_state_dict(best["model"])
reload_model.eval()
with torch.no_grad():
    torch.testing.assert_close(reference_features, reload_model.encode_images(x_check))
del reload_model
print(f"Checkpoint reload verified. Best reconstruction checkpoint: epoch {best['next_epoch']}.")
history_df = pd.DataFrame(history)
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(history_df.epoch, history_df.train_loss, label="Training (resampled images/masks)")
ax.plot(history_df.epoch, history_df.validation_loss, label="Validation (fixed images/masks)")
ax.set(xlabel="Epoch", ylabel="Masked-patch MSE", title=f"MAE loss — {MODE}")
ax.legend()
fig.tight_layout()
fig.savefig(RUN_DIR / "loss.png", dpi=130)
plt.show()

In [ ]:
visual_images, visual_noise = next(iter(make_loader(validation_paths[:4], fixed_masks=True)))
with torch.no_grad():
    _, visual_prediction, visual_mask = model(visual_images.to(DEVICE), visual_noise)
visual_prediction, visual_mask = visual_prediction.cpu(), visual_mask.cpu()
original_patches = patchify(visual_images)
normalized, original_mean, original_scale = normalized_targets(original_patches)
target = normalized if NORM_PIX_LOSS else original_patches
display_prediction = visual_prediction * original_scale + original_mean if NORM_PIX_LOSS else visual_prediction
# Visible patches are copied only for this composite visualization.
composite = torch.where(visual_mask.unsqueeze(-1).bool(), display_prediction, original_patches)
masked = unpatchify(original_patches * (1 - visual_mask.unsqueeze(-1)))
target_image, predicted_image = unpatchify(target), unpatchify(visual_prediction)
composite_image = unpatchify(composite)
fig, axes = plt.subplots(len(visual_images), 5, figsize=(13, 2.5*len(visual_images)), squeeze=False)
titles = ["Original", "75% masked", "Normalized target" if NORM_PIX_LOSS else "Raw target",
          "Decoder prediction (normalized)" if NORM_PIX_LOSS else "Decoder prediction (raw)",
          "Composite using ORIGINAL patch statistics" if NORM_PIX_LOSS else "Composite (raw pixels)"]
for row in range(len(visual_images)):
    arrays = [visual_images[row, 0], masked[row, 0], target_image[row, 0],
              predicted_image[row, 0], composite_image[row, 0]]
    for col, array in enumerate(arrays):
        low, high = (-2, 2) if NORM_PIX_LOSS and col in [2, 3] else (0, 1)
        axes[row, col].imshow(array, cmap="gray", vmin=low, vmax=high)
        axes[row, col].axis("off")
        if row == 0:
            axes[row, col].set_title(titles[col], fontsize=8, wrap=True)
fig.suptitle(f"Reconstruction diagnostics — {MODE}; visible-patch decoder outputs are not trained by the loss")
fig.tight_layout()
fig.savefig(RUN_DIR / "reconstructions.png", dpi=150)
plt.show()

## 6. Frozen study representations and the random-encoder control

For the pilot, every image from each labeled study is encoded without masking.
Mean pooling over patch tokens produces 192 numbers per image. We then accumulate
the mean and coordinate-wise maximum over images: 192 or 384 numbers per study.
Smoke mode uses only its two indexed images per study.

Mean pooling can dilute focal findings; maximum pooling can amplify artifacts
and depend on slice count. Neither is assumed superior. Both are computed in a
single encoder pass and compared with identical folds. No plane labels are used.
Feature files are keyed by encoder weights and manifest so stale caches are rejected.

In [ ]:
def state_digest(net):
    digest = hashlib.sha256()
    for name, tensor in sorted(net.state_dict().items()):
        digest.update(name.encode())
        digest.update(tensor.detach().cpu().contiguous().numpy().tobytes())
    return digest.hexdigest()


def extract_study_features(net, name):
    net.eval()
    net.zero_grad(set_to_none=True)
    net.requires_grad_(False)
    fingerprint = state_digest(net)
    cache = RUN_DIR / f"features_{name}.npz"
    if cache.exists():
        with np.load(cache, allow_pickle=False) as saved_features:
            assert str(saved_features["model_hash"]) == fingerprint
            assert str(saved_features["manifest_hash"]) == manifest_hash
            assert saved_features["study_ids"].tolist() == labeled_ids
            means, maxima = saved_features["mean"].copy(), saved_features["max"].copy()
    else:
        means, maxima = [], []
        with torch.inference_mode():
            for study_id in tqdm(labeled_ids, desc=f"Frozen features: {name}"):
                accumulator = torch.zeros(CFG["encoder_dim"], dtype=torch.float64)
                maximum = torch.full((CFG["encoder_dim"],), -torch.inf)
                count = 0
                for images in make_loader(study_paths[study_id]):
                    embeddings = net.encode_images(images.to(DEVICE)).cpu()
                    assert torch.isfinite(embeddings).all()
                    accumulator += embeddings.double().sum(dim=0)
                    maximum = torch.maximum(maximum, embeddings.max(dim=0).values)
                    count += len(images)
                assert count == len(study_paths[study_id]) and count > 0
                means.append((accumulator / count).float().numpy())
                maxima.append(maximum.numpy())
        means, maxima = np.stack(means), np.stack(maxima)
        np.savez_compressed(cache, study_ids=np.array(labeled_ids), mean=means, max=maxima,
                            counts=np.array([len(study_paths[s]) for s in labeled_ids]),
                            model_hash=fingerprint, manifest_hash=manifest_hash)
    assert state_digest(net) == fingerprint
    assert all(not p.requires_grad and p.grad is None for p in net.parameters())
    return {"mean": means, "mean_max": np.concatenate([means, maxima], axis=1)}


trained_features = extract_study_features(model, "mae")
random_model = SmallMAE(CFG).to(DEVICE)
random_model.load_state_dict(torch.load(initial_path, map_location=DEVICE, weights_only=True))
random_features = extract_study_features(random_model, "random")
encoder_hash_before_probes = state_digest(model)
random_hash_before_probes = state_digest(random_model)
representations = {"mae_mean": trained_features["mean"], "mae_mean_max": trained_features["mean_max"],
                   "random_mean": random_features["mean"], "random_mean_max": random_features["mean_max"]}
print({name: values.shape for name, values in representations.items()})

## 7. L2-regularized logistic probes, entirely in PyTorch

Five folds are assigned **once** and reused for all four representations. Each
target is a separate binary logistic regression. On each training fold we minimize

\[
\operatorname{mean}(\mathrm{BCEWithLogits}(Xw+b,y))
+\frac{\lVert w\rVert_2^2}{2 C n_{\mathrm{train}}},\qquad C=0.1.
\]

The intercept is not penalized. Features are standardized from training-fold
statistics only; near-constant dimensions use scale 1. L-BFGS runs on CPU in
float64 from zero initialization, with no dropout or class weighting. A small
gradient is the convergence criterion; every fit records objective, iterations,
function evaluations, and final gradient. Unconverged fits are visibly flagged.
No parameter or epoch selection uses the scored fold.

In [ ]:
Y = labeled[LABELS].to_numpy(dtype=np.float64)
splitter = MultilabelStratifiedKFold(n_splits=CFG["n_folds"], shuffle=True, random_state=SEED)
folds = list(splitter.split(np.zeros((len(Y), 1)), Y))
fold_assignment = np.full(len(Y), -1, dtype=int)
for fold, (train_idx, val_idx) in enumerate(folds):
    assert not (set(train_idx) & set(val_idx))
    assert (fold_assignment[val_idx] == -1).all()
    fold_assignment[val_idx] = fold
assert (fold_assignment >= 0).all()
pd.DataFrame({"study_id": labeled_ids, "fold": fold_assignment}).to_csv(RUN_DIR / "classification_folds.csv", index=False)


def fit_standardizer(train_features):
    mean = train_features.mean(axis=0)
    scale = train_features.std(axis=0, ddof=0)
    scale = np.where(scale > 1e-8, scale, 1.0)
    return mean, scale


def fit_probe(x, y):
    assert x.dtype == y.dtype == torch.float64 and x.device.type == "cpu"
    if torch.unique(y).numel() < 2:
        raise ValueError("A training fold has only one class: logistic intercept has no finite optimum.")
    head = nn.Linear(x.shape[1], 1, dtype=torch.float64)
    nn.init.zeros_(head.weight)
    nn.init.zeros_(head.bias)
    opt = torch.optim.LBFGS(head.parameters(), lr=1.0, max_iter=CFG["probe_max_iter"],
                            max_eval=CFG["probe_max_iter"] * 2,
                            tolerance_grad=CFG["probe_grad_tol"], tolerance_change=CFG["probe_change_tol"],
                            history_size=20, line_search_fn="strong_wolfe")

    def objective():
        logits = head(x).squeeze(1)
        return F.binary_cross_entropy_with_logits(logits, y) + head.weight.square().sum() / (2 * CFG["probe_C"] * len(y))

    def closure():
        opt.zero_grad(set_to_none=True)
        loss = objective()
        if not torch.isfinite(loss):
            raise RuntimeError("Nonfinite logistic objective")
        loss.backward()
        return loss

    opt.step(closure)
    final_loss = closure()
    gradient = max(p.grad.abs().max().item() for p in head.parameters())
    state = opt.state[head.weight]
    diagnostics = dict(objective=final_loss.item(), gradient_max=gradient,
                       iterations=int(state.get("n_iter", 0)),
                       function_evaluations=int(state.get("func_evals", 0)),
                       converged=gradient <= CFG["probe_grad_tol"])
    assert all(torch.isfinite(p).all() for p in head.parameters())
    return head.eval(), diagnostics


def safe_auc(y, probabilities):
    if np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, probabilities))


# Verify standardization really fits only its argument and handles constants.
toy_train = np.array([[0., 3.], [2., 3.]])
toy_val = np.array([[1e6, -1e6]])
toy_mean, toy_scale = fit_standardizer(toy_train)
np.testing.assert_allclose(toy_mean, [1., 3.])
np.testing.assert_allclose(toy_scale, [1., 1.])
assert np.isfinite((toy_val - toy_mean) / toy_scale).all()

all_diagnostics, all_metrics, fold_metrics = [], [], []
oof_by_representation = {}
for name, features in representations.items():
    oof = np.full(Y.shape, np.nan, dtype=np.float64)
    assignments = np.zeros(Y.shape, dtype=int)
    for fold, (train_idx, val_idx) in enumerate(folds):
        mean, scale = fit_standardizer(features[train_idx].astype(np.float64))
        np.testing.assert_allclose(mean, features[train_idx].astype(np.float64).mean(axis=0))
        x_train = torch.from_numpy((features[train_idx] - mean) / scale).double()
        x_val = torch.from_numpy((features[val_idx] - mean) / scale).double()
        weights, biases = [], []
        for target_idx, target_name in enumerate(LABELS):
            head, diagnostics = fit_probe(x_train, torch.from_numpy(Y[train_idx, target_idx]))
            with torch.no_grad():
                probabilities = head(x_val).squeeze(1).sigmoid().numpy()
            oof[val_idx, target_idx] = probabilities
            assignments[val_idx, target_idx] += 1
            weights.append(head.weight.detach().clone())
            biases.append(head.bias.detach().clone())
            all_diagnostics.append(dict(representation=name, fold=fold, target=target_name,
                                        train_positive=int(Y[train_idx, target_idx].sum()),
                                        val_positive=int(Y[val_idx, target_idx].sum()), **diagnostics))
            auc = safe_auc(Y[val_idx, target_idx], probabilities)
            fold_metrics.append(dict(representation=name, fold=fold, target=target_name,
                                     auc=auc, defined=bool(np.isfinite(auc)),
                                     positive=int(Y[val_idx, target_idx].sum()),
                                     negative=int(len(val_idx) - Y[val_idx, target_idx].sum())))
        torch.save(dict(weights=torch.cat(weights), biases=torch.cat(biases),
                        mean=torch.from_numpy(mean), scale=torch.from_numpy(scale), labels=LABELS,
                        train_studies=[labeled_ids[i] for i in train_idx],
                        validation_studies=[labeled_ids[i] for i in val_idx],
                        representation=name, C=CFG["probe_C"]), RUN_DIR / f"probe_{name}_fold{fold}.pt")
    assert (assignments == 1).all(), "Every study/target must be scored once"
    assert np.isfinite(oof).all() and ((oof >= 0) & (oof <= 1)).all()
    out = pd.DataFrame(oof, columns=LABELS)
    out.insert(0, "fold", fold_assignment)
    out.insert(0, "StudyInstanceUID", labeled_ids)
    out.to_csv(RUN_DIR / f"oof_{name}.csv", index=False)
    oof_by_representation[name] = oof
    for j, label in enumerate(LABELS):
        auc = safe_auc(Y[:, j], oof[:, j])
        all_metrics.append(dict(representation=name, target=label, auc=auc,
                                defined=bool(np.isfinite(auc)), positive=int(Y[:, j].sum()),
                                negative=int(len(Y) - Y[:, j].sum())))
    print(f"Completed {name}")

diagnostics_df = pd.DataFrame(all_diagnostics)
metrics_df = pd.DataFrame(all_metrics)
fold_metrics_df = pd.DataFrame(fold_metrics)
diagnostics_df.to_csv(RUN_DIR / "probe_diagnostics.csv", index=False)
metrics_df.to_csv(RUN_DIR / "target_metrics.csv", index=False)
fold_metrics_df.to_csv(RUN_DIR / "fold_metrics.csv", index=False)
unconverged = diagnostics_df.loc[~diagnostics_df.converged]
if len(unconverged):
    warnings.warn(f"{len(unconverged)} probe fits failed the gradient convergence criterion. Inspect diagnostics before interpreting scores.")
    display(unconverged)
assert state_digest(model) == encoder_hash_before_probes
assert state_digest(random_model) == random_hash_before_probes
assert all(not p.requires_grad and p.grad is None for p in model.parameters())
print("Frozen encoders, fold-only normalization and OOF coverage checks passed.")

## 8. Compare without overclaiming

The main report uses per-target AUC from pooled out-of-fold predictions, not an
average of tiny fold AUCs. Fold scores and counts remain available for inspection;
single-class AUCs are undefined, not zero. A partial macro average is flagged if
any target is undefined. Scores from different fold models can differ in scale.

Only 58 labeled studies are available, with few positives for some targets.
Small differences can be noise. These are four prespecified comparisons, not an
independent evaluation of whichever configuration looks best. Reconstruction
quality alone does not demonstrate useful diagnostic features.

Next experiments change one factor at a time: pooling (already compared here),
longer pretraining, normalized versus raw targets, then limited encoder fine-tuning.

In [ ]:
summary = metrics_df.groupby("representation", sort=False).agg(
    macro_auc=("auc", "mean"), defined_targets=("defined", "sum"))
summary["all_targets_defined"] = summary.defined_targets == len(LABELS)
summary["unconverged_fits"] = diagnostics_df.groupby("representation").converged.apply(lambda s: int((~s).sum()))
summary["mode"] = MODE
summary["interpretation"] = "PIPELINE CHECK ONLY" if MODE == "smoke" else "Exploratory out-of-fold estimate"
summary.to_csv(RUN_DIR / "comparison.csv")
print("SMOKE RESULTS — NOT PERFORMANCE ESTIMATES" if MODE == "smoke" else "Exploratory five-fold results")
display(summary)
display(metrics_df.pivot(index="target", columns="representation", values="auc").reindex(LABELS))
display(fold_metrics_df.loc[~fold_metrics_df.defined])

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(summary.index, summary.macro_auc)
ax.axhline(0.5, color="gray", linestyle="--", label="Chance ranking reference")
ax.set(ylim=(0, 1), ylabel="Pooled OOF macro ROC-AUC",
       title="PIPELINE CHECK ONLY" if MODE == "smoke" else "Exploratory comparisons; no independent model selection")
ax.tick_params(axis="x", rotation=15)
ax.legend()
fig.tight_layout()
fig.savefig(RUN_DIR / "comparison.png", dpi=140)
plt.show()

verification = dict(mode=MODE, patch_round_trip=True, mask_positions_and_counts=True,
                    constant_patch_normalization=True, masked_only_loss=True, finite_gradients=True,
                    schedule=True, checkpoint_reload=True, frozen_encoders=True,
                    train_fold_standardization=True, complete_oof_coverage=True,
                    probe_fits=len(diagnostics_df), unconverged_fits=len(unconverged),
                    complete_pilot_executed=MODE == "pilot")
(RUN_DIR / "verification.json").write_text(json.dumps(verification, indent=2))
print(f"Artifacts saved to {RUN_DIR.resolve()}")
print("To run the full pilot: choose MODE='pilot', ALLOW_PILOT=True and a new run directory.")